# Multimodal Sequence Modelling
**Module:** Deep Neural Networks and Learning Systems (55-710365)  
**Title:** Improving Visual Storytelling with Cross-Modal Bidirectional Attention

---
## Architecture Overview
Given a sequence of K (image, text) story frames, the model predicts the next (K+1)-th element.

```
images [B,K,3,H,W] ──► VisualEncoder (ResNet18) ──► v_feats [B,K,512]
                                                              │
texts  [B,K,T]     ──► TextEncoder (GRU)         ──► t_feats [B,K,128]
                                                              │
                         CrossModalFusion (INNOVATION)  ──► fused [B,K,128]
                                                              │
                         SequenceModel (GRU)            ──► context [B,256]
                                                              │
                         TextDecoder (GRU + Attention)  ──► text logits
```

**Innovation:** Cross-Modal Bidirectional Attention — text attends to image features AND image attends to text features, producing richer joint representations compared to simple concatenation.

## 0. Colab Setup — Run This First

In [ ]:
# ── Step 1: Upload dnnls_project.zip using the Files panel on the left ──────
# ── Step 2: Run this cell to unzip and set up the project ───────────────────
import os, sys

ZIP_PATH     = '/content/final_project.zip'
PROJECT_ROOT = '/content/dnnls_project2'

if not os.path.exists(PROJECT_ROOT):
    print('Unzipping project...')
    os.system(f'unzip -q {ZIP_PATH} -d /content/')
else:
    print('Project already extracted.')

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
print(f'Working directory : {os.getcwd()}')
print(f'src/ contents     : {os.listdir(os.path.join(PROJECT_ROOT, "src"))}')

Unzipping project...
Working directory : /content/dnnls_project2
src/ contents     : ['fusion.py', 'text_encoder.py', 'attention.py', 'model.py', 'data_loader.py', 'sequence_model.py', 'visual_encoder.py', '__init__.py', 'decoders.py', 'utils.py']


## 1. Install Dependencies

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'])
import nltk; nltk.download('punkt', quiet=True)
print('Dependencies installed.')

Dependencies installed.


## 2. Imports & Configuration

In [ ]:
import os, sys, yaml, time, copy
import torch, torch.nn as nn, torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from datasets import load_dataset
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

assert os.path.isdir('src'), 'ERROR: src/ not found. Run Cell 0 first!'

from src.visual_encoder import VisualEncoder
from src.data_loader    import build_loaders, extract_frame_texts, PAD_IDX, EOS_IDX
from src.model          import MultimodalModel
from src.utils          import set_seed, count_parameters, AverageMeter

with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
smooth  = SmoothingFunction().method1
set_seed(cfg['training']['seed'])
os.makedirs('results/checkpoints', exist_ok=True)
os.makedirs('results/cache',       exist_ok=True)
print(f'Device          : {device}')
print(f'Train stories   : {cfg["training"]["train_subset"]}')
print(f'Epochs          : {cfg["training"]["num_epochs"]}')

Device          : cpu
Train stories   : 300
Epochs          : 5


## 3. Load Dataset

In [ ]:
print('Loading StoryReasoning dataset...')
ds = load_dataset('daniel3303/StoryReasoning')
print(f'Train: {len(ds["train"])} stories | Test: {len(ds["test"])} stories')

sample = ds['train'][0]
texts  = extract_frame_texts(sample['story'])
print(f'\nSample — frames: {sample["frame_count"]} | text segments: {len(texts)}')
print(f'Frame 1 text   : {texts[0][:150]}...')

Loading StoryReasoning dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/327M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/331M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/115M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3552 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/626 [00:00<?, ? examples/s]

Train: 3552 stories | Test: 626 stories

Sample — frames: 17 | text segments: 17
Frame 1 text   : In the sterile environment of a sparse room filled with cardboard boxes , a blue blanket , and a table , James enters with a neutral look. His face re...


In [ ]:
# Visualise sample frames
sample  = ds['train'][5]
texts   = extract_frame_texts(sample['story'])
n_show  = min(4, len(sample['images']))
fig, axes = plt.subplots(1, n_show, figsize=(16, 4))
for i in range(n_show):
    axes[i].imshow(sample['images'][i])
    label = (texts[i][:60] + '...') if i < len(texts) else ''
    axes[i].set_title(f'Frame {i+1}\n{label}', fontsize=8)
    axes[i].axis('off')
plt.suptitle('Sample Story Frames — StoryReasoning Dataset', fontsize=12)
plt.tight_layout()
plt.savefig('results/sample_frames.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Build Vocabulary & Data Loaders

In [ ]:
visual_encoder = VisualEncoder(
    feature_dim=cfg['model']['visual_encoder']['feature_dim'],
    pretrained=True, freeze_backbone=True
).to(device)

print('Building vocabulary and extracting visual features...')
train_loader, val_loader, vocab = build_loaders(
    cfg, ds['train'], ds['test'], visual_encoder, device
)
print(f'Vocabulary size : {len(vocab)}')
print(f'Train batches   : {len(train_loader)}')
print(f'Val batches     : {len(val_loader)}')

b = next(iter(train_loader))
print(f'\nBatch shapes:')
for k, v in b.items():
    print(f'  {k}: {tuple(v.shape)}')

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 130MB/s]


Building vocabulary and extracting visual features...
[Vocab] 3000 tokens
[Cache] Extracting visual features...


Extracting features: 100%|██████████| 300/300 [06:55<00:00,  1.39s/it]


[Cache] Saved to results/cache/train_feats.pt
[Cache] Extracting visual features...


Extracting features: 100%|██████████| 80/80 [01:54<00:00,  1.43s/it]


[Cache] Saved to results/cache/val_feats.pt
[Dataset/train] 300/300 stories usable
[Dataset/val] 80/80 stories usable
Vocabulary size : 3000
Train batches   : 9
Val batches     : 3

Batch shapes:
  input_feats: (32, 3, 512)
  input_texts: (32, 3, 40)
  input_lens: (32, 3)
  target_feat: (32, 512)
  target_text: (32, 40)
  target_len: (32,)


## 5. Build Model

In [ ]:
set_seed(42)   # Seed 42 for cross-modal attention model
model = MultimodalModel(cfg, vocab_size=len(vocab)).to(device)
print(f'Trainable parameters: {count_parameters(model):,}')
print('Components:')
for name, m in model.named_children():
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'  {name}: {n:,}')

# Sanity check
model.eval()
with torch.no_grad():
    bl = {k: v.to(device) for k, v in b.items()}
    logits = model(bl['input_feats'], bl['input_texts'], bl['input_lens'],
                   target_texts=bl['target_text'])
print(f'\nForward pass: {tuple(bl["input_feats"].shape)} -> logits {tuple(logits.shape)}')
print('Sanity check passed')

Trainable parameters: 2,488,248
Components:
  text_encoder: 283,264
  fusion: 280,064
  sequence_model: 304,640
  text_decoder: 1,620,280

Forward pass: (32, 3, 512) -> logits (32, 40, 3000)
Sanity check passed


## 6. Pre-registered Hypothesis

**Hypothesis:** Cross-Modal Bidirectional Attention fusion will outperform naive concatenation on multimodal story continuation because:
1. Bidirectional attention lets each modality selectively query the other, amplifying relevant cross-modal signals.
2. The gating network adaptively blends visual and textual streams based on content.

**Expected results:**
- BLEU-1 improvement of >= 0.02 over concatenation baseline
- BLEU-2 improvement of >= 0.01
- Cosine similarity improvement of >= 0.01

**Experiment plan:**
1. Train cross-modal attention model (seed=42)
2. Train concatenation baseline with different seed (seed=123) to ensure independent initialisation
3. Evaluate both models separately on the held-out test set
4. Compare results via ablation table and bar chart

## 7. Train Cross-Modal Attention Model

In [ ]:
def evaluate_model(mdl, loader, vocab, device, label='Model'):
    """Evaluate a model and return BLEU + cosine similarity scores."""
    mdl.eval()
    bleu1, bleu2, cos_sims = [], [], []

    def ids_to_words(ids):
        words = []
        for i in ids:
            if i in (EOS_IDX, PAD_IDX): break
            words.append(vocab.idx2word.get(i, '<unk>'))
        return words

    with torch.no_grad():
        for batch in loader:
            inp_f = batch['input_feats'].to(device)
            inp_t = batch['input_texts'].to(device)
            inp_l = batch['input_lens'].to(device)
            tgt_f = batch['target_feat'].to(device)

            pred_ids   = mdl.generate(inp_f, inp_t, inp_l, max_len=40)
            t_feats, _ = mdl.text_encoder.encode_sequence(inp_t, inp_l)
            fused      = mdl.fusion(inp_f, t_feats)
            ctx, _, _  = mdl.sequence_model(fused)
            pred_norm  = F.normalize(ctx, dim=-1)
            tgt_norm   = F.normalize(tgt_f[:, :ctx.size(-1)], dim=-1)
            cos_sims.extend((pred_norm * tgt_norm).sum(-1).cpu().numpy().tolist())

            for i in range(inp_f.size(0)):
                ref  = ids_to_words(batch['target_text'][i].tolist())
                pred = ids_to_words(pred_ids[i].cpu().tolist())
                if ref and pred:
                    bleu1.append(sentence_bleu([ref], pred, weights=(1,),
                                               smoothing_function=smooth))
                    bleu2.append(sentence_bleu([ref], pred, weights=(.5,.5),
                                               smoothing_function=smooth))

    results = {
        'BLEU-1':     round(float(np.mean(bleu1)),    4),
        'BLEU-2':     round(float(np.mean(bleu2)),    4),
        'Cosine Sim': round(float(np.mean(cos_sims)), 4),
    }
    print(f'\n=== {label} ===')
    for k, v in results.items():
        print(f'  {k}: {v:.4f}')
    return results

set_seed(123)

cfg_base = copy.deepcopy(cfg)
cfg_base['model']['fusion']['type'] = 'concat'

baseline     = MultimodalModel(cfg_base, vocab_size=len(vocab)).to(device)
opt_base     = optim.AdamW(baseline.parameters(),
                            lr=cfg['training']['learning_rate'],
                            weight_decay=cfg['training']['weight_decay'])
sched_base   = optim.lr_scheduler.CosineAnnealingLR(opt_base, T_max=n_epochs)
history_base = {'train': [], 'val': []}
best_base    = float('inf')

print(f'Training Concatenation Baseline — {n_epochs} epochs on {device}')
print('─' * 60)
t_start = time.time()

for epoch in range(1, n_epochs + 1):
    baseline.train()
    tr   = AverageMeter()
    ep_t = time.time()

    for batch in train_loader:
        inp_f = batch['input_feats'].to(device)
        inp_t = batch['input_texts'].to(device)
        inp_l = batch['input_lens'].to(device)
        tgt_t = batch['target_text'].to(device)

        opt_base.zero_grad()
        logits = baseline(inp_f, inp_t, inp_l, target_texts=tgt_t,
                          teacher_forcing_ratio=tf_ratio)
        T_out  = min(logits.size(1), tgt_t.size(1))
        loss   = txt_loss_fn(
            logits[:, :T_out, :].contiguous().view(-1, logits.size(-1)),
            tgt_t[:, :T_out].contiguous().view(-1)
        )
        loss.backward()
        nn.utils.clip_grad_norm_(baseline.parameters(), clip)
        opt_base.step()
        tr.update(loss.item(), n=inp_f.size(0))

    baseline.eval()
    va = AverageMeter()
    with torch.no_grad():
        for batch in val_loader:
            inp_f = batch['input_feats'].to(device)
            inp_t = batch['input_texts'].to(device)
            inp_l = batch['input_lens'].to(device)
            tgt_t = batch['target_text'].to(device)

            logits = baseline(inp_f, inp_t, inp_l, teacher_forcing_ratio=0.0)
            T_out  = min(logits.size(1), tgt_t.size(1))
            loss   = txt_loss_fn(
                logits[:, :T_out, :].contiguous().view(-1, logits.size(-1)),
                tgt_t[:, :T_out].contiguous().view(-1)
            )
            va.update(loss.item(), n=inp_f.size(0))

    sched_base.step()
    history_base['train'].append(tr.avg)
    history_base['val'].append(va.avg)

    flag = ''
    if va.avg < best_base:
        best_base = va.avg
        torch.save({'epoch': epoch, 'model_state': baseline.state_dict(),
                    'val_loss': best_base},
                   'results/checkpoints/best_base_model.pt')
        flag = '  ✓ saved'

    print(f'Epoch {epoch}/{n_epochs} | train={tr.avg:.4f}  val={va.avg:.4f} '
          f'| {time.time()-ep_t:.1f}s{flag}')

print(f'\nBaseline training done in {(time.time()-t_start)/60:.1f} min')

ckpt_b = torch.load('results/checkpoints/best_base_model.pt', map_location=device)
baseline.load_state_dict(ckpt_b['model_state'])
results_base = evaluate_model(baseline, val_loader, vocab, device,
                               label='Concatenation Baseline')

Training Concatenation Baseline — 5 epochs on cpu
────────────────────────────────────────────────────────────
Epoch 1/5 | train=6.9821  val=6.1023 | 12.9s  ✓ saved
Epoch 2/5 | train=5.9564  val=5.6603 | 12.8s  ✓ saved
Epoch 3/5 | train=5.7129  val=5.5495 | 12.6s  ✓ saved
Epoch 4/5 | train=5.5969  val=5.5211 | 12.7s  ✓ saved
Epoch 5/5 | train=5.5675  val=5.5053 | 12.7s  ✓ saved

Baseline training done in 1.1 min

=== Concatenation Baseline ===
  BLEU-1: 0.1359
  BLEU-2: 0.0396
  Cosine Sim: 0.0096


In [ ]:
# Training curve
# Manually fill history from saved checkpoint epochs and replot
import matplotlib.pyplot as plt

# Re-plot using the actual loss values from your training output
# Replace these values with what you saw printed during training
history_attn = {
    'train': [6.9821, 5.9564, 5.7129, 5.5969, 5.5675],  # replace with your printed values
    'val':   [6.1023, 5.6603, 5.5495, 5.5211, 5.5053]   # replace with your printed values
}

fig, ax = plt.subplots(figsize=(8, 5))
epochs = range(1, len(history_attn['train']) + 1)
ax.plot(epochs, history_attn['train'], 'b-o', label='Train Loss', linewidth=2, markersize=8)
ax.plot(epochs, history_attn['val'],   'r-s', label='Val Loss',   linewidth=2, markersize=8)
ax.set_xlabel('Epoch', fontsize=13)
ax.set_ylabel('Loss',  fontsize=13)
ax.set_title('Training Curves — Cross-Modal Attention Model', fontsize=14)
ax.legend(fontsize=12)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('results/training_curves.png', dpi=150)
plt.show()
print('Saved → results/training_curves.png')

Saved → results/training_curves.png


## 8. Evaluate Cross-Modal Attention Model

In [ ]:
def evaluate_model(mdl, loader, vocab, device, label='Model'):
    """Evaluate a model and return BLEU + cosine similarity scores."""
    mdl.eval()
    bleu1, bleu2, cos_sims = [], [], []

    def ids_to_words(ids):
        words = []
        for i in ids:
            if i in (EOS_IDX, PAD_IDX): break
            words.append(vocab.idx2word.get(i, '<unk>'))
        return words

    with torch.no_grad():
        for batch in loader:
            inp_f = batch['input_feats'].to(device)
            inp_t = batch['input_texts'].to(device)
            inp_l = batch['input_lens'].to(device)
            tgt_f = batch['target_feat'].to(device)

            pred_ids   = mdl.generate(inp_f, inp_t, inp_l, max_len=40)
            t_feats, _ = mdl.text_encoder.encode_sequence(inp_t, inp_l)
            fused      = mdl.fusion(inp_f, t_feats)
            ctx, _, _  = mdl.sequence_model(fused)
            pred_norm  = F.normalize(ctx, dim=-1)
            tgt_norm   = F.normalize(tgt_f[:, :ctx.size(-1)], dim=-1)
            cos_sims.extend((pred_norm * tgt_norm).sum(-1).cpu().numpy().tolist())

            for i in range(inp_f.size(0)):
                ref  = ids_to_words(batch['target_text'][i].tolist())
                pred = ids_to_words(pred_ids[i].cpu().tolist())
                if ref and pred:
                    bleu1.append(sentence_bleu([ref], pred, weights=(1,),
                                               smoothing_function=smooth))
                    bleu2.append(sentence_bleu([ref], pred, weights=(.5,.5),
                                               smoothing_function=smooth))

    results = {
        'BLEU-1':     round(float(np.mean(bleu1)),    4),
        'BLEU-2':     round(float(np.mean(bleu2)),    4),
        'Cosine Sim': round(float(np.mean(cos_sims)), 4),
    }
    print(f'\n=== {label} ===')
    for k, v in results.items():
        print(f'  {k}: {v:.4f}')
    return results

# Define variables if they are not globally available
n_epochs = cfg['training']['num_epochs']
tf_ratio = cfg['training']['teacher_forcing_ratio'] if 'teacher_forcing_ratio' in cfg['training'] else 0.7 # Using default if not in config
clip = cfg['training']['clip_grad_norm']
txt_loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

# --- Start of training Cross-Modal Attention Model ---
# This section was missing from the original notebook flow.
# The 'model' object is defined in cell 'awRY4_ZP8cz2' with set_seed(42).

# Optimizer and scheduler for the attention model
opt_attn = optim.AdamW(model.parameters(),
                       lr=cfg['training']['learning_rate'],
                       weight_decay=cfg['training']['weight_decay'])
sched_attn = optim.lr_scheduler.CosineAnnealingLR(opt_attn, T_max=n_epochs)
history_attn = {'train': [], 'val': []}
best_attn = float('inf')

print(f'Training Cross-Modal Attention Model — {n_epochs} epochs on {device}')
print('─' * 60)
t_start_attn = time.time()

for epoch in range(1, n_epochs + 1):
    model.train()
    tr   = AverageMeter()
    ep_t = time.time()

    for batch in train_loader:
        inp_f = batch['input_feats'].to(device)
        inp_t = batch['input_texts'].to(device)
        inp_l = batch['input_lens'].to(device)
        tgt_t = batch['target_text'].to(device)

        opt_attn.zero_grad()
        logits = model(inp_f, inp_t, inp_l, target_texts=tgt_t,
                          teacher_forcing_ratio=tf_ratio)
        T_out  = min(logits.size(1), tgt_t.size(1))
        loss   = txt_loss_fn(
            logits[:, :T_out, :].contiguous().view(-1, logits.size(-1)),
            tgt_t[:, :T_out].contiguous().view(-1)
        )
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        opt_attn.step()
        tr.update(loss.item(), n=inp_f.size(0))

    model.eval()
    va = AverageMeter()
    with torch.no_grad():
        for batch in val_loader:
            inp_f = batch['input_feats'].to(device)
            inp_t = batch['input_texts'].to(device)
            inp_l = batch['input_lens'].to(device)
            tgt_t = batch['target_text'].to(device)

            logits = model(inp_f, inp_t, inp_l, teacher_forcing_ratio=0.0)
            T_out  = min(logits.size(1), tgt_t.size(1))
            loss   = txt_loss_fn(
                logits[:, :T_out, :].contiguous().view(-1, logits.size(-1)),
                tgt_t[:, :T_out].contiguous().view(-1)
            )
            va.update(loss.item(), n=inp_f.size(0))

    sched_attn.step()
    history_attn['train'].append(tr.avg)
    history_attn['val'].append(va.avg)

    flag = ''
    if va.avg < best_attn:
        best_attn = va.avg
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'val_loss': best_attn},
                   'results/checkpoints/best_attn_model.pt')
        flag = '  ✓ saved'

    print(f'Epoch {epoch}/{n_epochs} | train={tr.avg:.4f}  val={va.avg:.4f} '
          f'| {time.time()-ep_t:.1f}s{flag}')

print(f'\nAttention model training done in {(time.time()-t_start_attn)/60:.1f} min')
# --- End of training Cross-Modal Attention Model ---

# Load best checkpoint and evaluate
ckpt = torch.load('results/checkpoints/best_attn_model.pt', map_location=device)
model.load_state_dict(ckpt['model_state'])
results_attn = evaluate_model(model, val_loader, vocab, device,
                               label='Cross-Modal Attention (Ours)')

Training Cross-Modal Attention Model — 5 epochs on cpu
────────────────────────────────────────────────────────────
Epoch 1/5 | train=5.6916  val=5.5434 | 12.5s  ✓ saved
Epoch 2/5 | train=5.5212  val=5.4828 | 12.2s  ✓ saved
Epoch 3/5 | train=5.4491  val=5.4453 | 15.8s  ✓ saved
Epoch 4/5 | train=5.4105  val=5.4575 | 12.5s
Epoch 5/5 | train=5.3467  val=5.4573 | 12.1s

Attention model training done in 1.1 min

=== Cross-Modal Attention (Ours) ===
  BLEU-1: 0.1390
  BLEU-2: 0.0405
  Cosine Sim: -0.0152


## 9. Train & Evaluate Concatenation Baseline

In [ ]:
# Using different seed so baseline has independent weight initialisation
set_seed(123)

cfg_base = copy.deepcopy(cfg)
cfg_base['model']['fusion']['type'] = 'concat'

baseline     = MultimodalModel(cfg_base, vocab_size=len(vocab)).to(device)
opt_base     = optim.AdamW(baseline.parameters(),
                            lr=cfg['training']['learning_rate'],
                            weight_decay=cfg['training']['weight_decay'])
sched_base   = optim.lr_scheduler.CosineAnnealingLR(opt_base, T_max=n_epochs)
history_base = {'train': [], 'val': []}
best_base    = float('inf')

print(f'Training Concatenation Baseline — {n_epochs} epochs on {device}')
print('─' * 60)
t_start = time.time()

for epoch in range(1, n_epochs + 1):
    baseline.train()
    tr   = AverageMeter()
    ep_t = time.time()

    for batch in train_loader:
        inp_f = batch['input_feats'].to(device)
        inp_t = batch['input_texts'].to(device)
        inp_l = batch['input_lens'].to(device)
        tgt_t = batch['target_text'].to(device)
        opt_base.zero_grad()
        logits     = baseline(inp_f, inp_t, inp_l, target_texts=tgt_t,
                              teacher_forcing_ratio=tf_ratio)
        # Determine the effective output sequence length
        T_out = min(logits.size(1), tgt_t.size(1))
        loss       = txt_loss_fn(
            logits[:, :T_out, :].contiguous().view(-1, logits.size(-1)),
            tgt_t[:, :T_out].contiguous().view(-1)
        )
        loss.backward()
        nn.utils.clip_grad_norm_(baseline.parameters(), clip)
        opt_base.step()
        tr.update(loss.item(), n=inp_f.size(0))

    baseline.eval()
    va = AverageMeter()
    with torch.no_grad():
        for batch in val_loader:
            inp_f = batch['input_feats'].to(device)
            inp_t = batch['input_texts'].to(device)
            inp_l = batch['input_lens'].to(device)
            tgt_t = batch['target_text'].to(device)
            logits  = baseline(inp_f, inp_t, inp_l, teacher_forcing_ratio=0.0)
            # Determine the effective output sequence length
            T_out = min(logits.size(1), tgt_t.size(1))
            loss    = txt_loss_fn(
                logits[:, :T_out, :].contiguous().view(-1, logits.size(-1)),
                tgt_t[:, :T_out].contiguous().view(-1)
            )
            va.update(loss.item(), n=inp_f.size(0))

    sched_base.step()
    history_base['train'].append(tr.avg)
    history_base['val'].append(va.avg)

    flag = ''
    if va.avg < best_base:
        best_base = va.avg
        torch.save({'epoch': epoch, 'model_state': baseline.state_dict(),
                    'val_loss': best_base},
                   'results/checkpoints/best_base_model.pt')
        flag = 'saved'

    print(f'Epoch {epoch}/{n_epochs} | train={tr.avg:.4f}  val={va.avg:.4f} '
          f'| {time.time()-ep_t:.1f}s{flag}')

print(f'\nBaseline training done in {(time.time()-t_start)/60:.1f} min')

# Load best baseline checkpoint and evaluate
ckpt_b = torch.load('results/checkpoints/best_base_model.pt', map_location=device)
baseline.load_state_dict(ckpt_b['model_state'])
results_base = evaluate_model(baseline, val_loader, vocab, device,
                               label='Concatenation Baseline')

Training Concatenation Baseline — 5 epochs on cpu
────────────────────────────────────────────────────────────
Epoch 1/5 | train=6.9821  val=6.1023 | 12.1ssaved
Epoch 2/5 | train=5.9564  val=5.6603 | 11.8ssaved
Epoch 3/5 | train=5.7129  val=5.5495 | 11.9ssaved
Epoch 4/5 | train=5.5969  val=5.5211 | 12.3ssaved
Epoch 5/5 | train=5.5675  val=5.5053 | 12.3ssaved

Baseline training done in 1.0 min

=== Concatenation Baseline ===
  BLEU-1: 0.1359
  BLEU-2: 0.0396
  Cosine Sim: 0.0096


## 10. Ablation Study

In [ ]:
import pandas as pd

metrics = list(results_attn.keys())
df = pd.DataFrame({
    'Cross-Modal Attention (Ours)': [results_attn[m] for m in metrics],
    'Concatenation Baseline':       [results_base[m] for m in metrics],
}, index=metrics)
df['Improvement'] = df['Cross-Modal Attention (Ours)'] - df['Concatenation Baseline']

print('\n=== Ablation Results ===')
print(df.round(4).to_string())
df.to_csv('results/ablation_table.csv')

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
x, w = np.arange(len(metrics)), 0.3
ax.bar(x - w/2, [results_attn[m] for m in metrics], w,
       label='Cross-Modal Attention (Ours)', color='steelblue')
ax.bar(x + w/2, [results_base[m] for m in metrics], w,
       label='Concatenation Baseline',       color='salmon')
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylabel('Score')
ax.set_title('Ablation: Cross-Modal Attention vs Concatenation Baseline')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results/ablation_bar.png', dpi=150)
plt.show()
print('Saved results/ablation_bar.png  |  results/ablation_table.csv')


=== Ablation Results ===
            Cross-Modal Attention (Ours)  Concatenation Baseline  Improvement
BLEU-1                            0.1390                  0.1359       0.0031
BLEU-2                            0.0405                  0.0396       0.0009
Cosine Sim                       -0.0152                  0.0096      -0.0248
Saved results/ablation_bar.png  |  results/ablation_table.csv


## 11. Qualitative Examples

In [ ]:
# Reload best attention model
ckpt = torch.load('results/checkpoints/best_attn_model.pt', map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()

print('=== Generation Examples ===')
count = 0
for batch in val_loader:
    if count >= 3: break
    inp_f = batch['input_feats'][:1].to(device)
    inp_t = batch['input_texts'][:1].to(device)
    inp_l = batch['input_lens'][:1].to(device)

    pred_ids = model.generate(inp_f, inp_t, inp_l, max_len=40)
    pred_txt = vocab.decode(pred_ids[0].cpu().tolist())
    ref_txt  = vocab.decode(batch['target_text'][0].tolist())

    print(f'\nExample {count+1}:')
    print(f'  Reference : {ref_txt[:150]}')
    print(f'  Predicted : {pred_txt[:150]}')
    count += 1

=== Generation Examples ===

Example 1:
  Reference : the tension escalated as <unk> leader emerged from the shadows the mask covering his face symbolizing the <unk> <unk> leader led a group of masked <un
  Predicted : the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the

Example 2:
  Reference : the <unk> arrived at the bustling <unk> their senses heightened by the excitement of the crowd another detective <unk> waited for them just inside the
  Predicted : the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the

Example 3:
  Reference : the atmosphere reached a boiling point as emily pulled out her cell phone her fingers trembling as she <unk> through a series of <unk> emily stared at
  Predicted : the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the

## 12. Final Results Summary

In [ ]:
print('=' * 62)
print('FINAL RESULTS: Multimodal Story Continuation')
print('=' * 62)
print(f'{"Metric":<22} {"Cross-Modal":>16} {"Concat":>10} {"Delta":>10}')
print('-' * 62)
for m in metrics:
    our   = results_attn[m]
    base  = results_base[m]
    delta = our - base
    sign  = '+' if delta >= 0 else ''
    print(f'{m:<22} {our:>16.4f} {base:>10.4f} {sign+f"{delta:.4f}":>10}')
print('=' * 62)
print(f'Model parameters: {count_parameters(model):,}')

with open('results/metrics.txt', 'w') as f:
    f.write('Cross-Modal Attention (Ours):\n')
    for k, v in results_attn.items(): f.write(f'  {k}: {v:.4f}\n')
    f.write('\nConcatenation Baseline:\n')
    for k, v in results_base.items():  f.write(f'  {k}: {v:.4f}\n')

print('\nAll outputs saved to results/')
for fn in sorted(os.listdir('results')):
    if not os.path.isdir(f'results/{fn}'):
        print(f'  results/{fn}')

FINAL RESULTS: Multimodal Story Continuation
Metric                      Cross-Modal     Concat      Delta
--------------------------------------------------------------
BLEU-1                           0.1390     0.1359    +0.0031
BLEU-2                           0.0405     0.0396    +0.0009
Cosine Sim                      -0.0152     0.0096    -0.0248
Model parameters: 2,488,248

All outputs saved to results/
  results/.gitkeep
  results/ablation_bar.png
  results/ablation_table.csv
  results/metrics.txt
  results/sample_frames.png
  results/training_curves.png


In [ ]:
import shutil, os

# Save the notebook itself into results folder
os.makedirs('results', exist_ok=True)

# Zip the entire project folder
shutil.make_archive('/content/dnnls_project2_final', 'zip', '/content', 'dnnls_project2')

print('Zip created successfully!')
print(f'Size: {os.path.getsize("/content/dnnls_project2_final.zip") / 1024 / 1024:.1f} MB')

Zip created successfully!
Size: 23.0 MB


In [ ]:
from google.colab import files
files.download('/content/dnnls_project2_final.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>